# Phase 0 dry run: Qwen3.5-0.8B operator agent

Proves the chain end to end before any credits are spent:
**verified data -> fine-tune -> GGUF -> llama.cpp -> Blender.**

- Data: `domains/operator_agent/data/dryrun_train.jsonl` (500 harness-verified op/1 examples)
- Model: `unsloth/Qwen3.5-0.8B`, LoRA, text layers only (the model is natively multimodal; vision layers stay frozen)
- Output: a merged `q8_0` GGUF (~0.8GB) to run with `llama-server`

Runs on free **Colab (T4)** or **Kaggle (T4 x2 / P100)**. Expect ~5 minutes of training.

The success bar is not accuracy on 500 examples -- it is that the GGUF loads in
llama.cpp and answers in valid op/1 that Blender executes. Score it locally with
`scripts/eval_operator_agent.py` afterwards.

## 1. Install

In [ ]:
%%capture
!pip install --upgrade unsloth

## 2. Load the dataset

Upload `dryrun_train.jsonl` when prompted (Colab), or add it as a Kaggle dataset
and point `DATASET_PATH` at `/kaggle/input/...`.

In [ ]:
import json, os
from pathlib import Path

DATASET_PATH = Path("dryrun_train.jsonl")
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ  # not os.path.exists("/kaggle"): Colab has that folder too

if ON_KAGGLE:
    found = list(Path("/kaggle/input").rglob("dryrun_train.jsonl"))
    assert found, "Add dryrun_train.jsonl as a Kaggle dataset first"
    DATASET_PATH = found[0]
elif not DATASET_PATH.exists():
    from google.colab import files
    uploaded = files.upload()
    DATASET_PATH = Path(next(iter(uploaded)))

records = [json.loads(line) for line in DATASET_PATH.read_text().splitlines() if line.strip()]
print(len(records), "examples")
print(records[0]["messages"][1]["content"][:300])
print("->", records[0]["messages"][2]["content"])

## 3. Load the model and attach LoRA

In [ ]:
from unsloth import FastModel
import torch

MODEL_NAME = "unsloth/Qwen3.5-0.8B"
MAX_SEQ_LENGTH = 1024  # longest dry-run example is well under 512 tokens

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,     # 0.8B fits in 16-bit on a T4; avoids quantization noise in a tiny model
    full_finetuning=False,
)
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    random_state=3407,
)

## 4. Format with the chat template

Thinking is disabled: the operator agent must answer with JSON immediately, and
the same `enable_thinking=False` has to be used at inference (llama-server:
`--chat-template-kwargs '{"enable_thinking": false}'`).

In [ ]:
from datasets import Dataset

text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)  # multimodal processor -> its text tokenizer

def to_text(record):
    return {"text": text_tokenizer.apply_chat_template(
        record["messages"], tokenize=False, add_generation_prompt=False, enable_thinking=False)}

dataset = Dataset.from_list([to_text(r) for r in records])
split = dataset.train_test_split(test_size=25, seed=3407)
train_dataset, eval_dataset = split["train"], split["test"]
print(train_dataset[0]["text"])

## 5. Train (loss on the assistant reply only)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=text_tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=1,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_steps=10,
        lr_scheduler_type="linear",
        logging_steps=10,
        eval_strategy="epoch",
        optim="adamw_8bit",
        weight_decay=0.01,
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
stats = trainer.train()
print(stats)

## 6. Quick check: does it answer in op/1?

In [ ]:
import re

FastModel.for_inference(model)
OP_RE = re.compile(r"^[a-z_][a-z0-9_]*\.[a-z_][a-z0-9_]*$")

def generate(messages):
    prompt = text_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = text_tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=160, do_sample=False)
    return text_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

valid = 0
for record in records[-10:]:  # training rows: this checks the format was learned, not accuracy
    reply = generate(record["messages"][:2])
    try:
        calls = json.loads(reply)["calls"]
        ok = all(OP_RE.match(c["op"]) for c in calls)
    except Exception:
        ok = False
    valid += ok
    print("OK " if ok else "BAD", record["request"], "->", reply)
print(f"\n{valid}/10 valid op/1")

## 7. Export to GGUF and download

Merges the LoRA into the base and writes `q8_0`. For the dry run a merged model
is simplest; per-specialist adapters on the 27B come in Phase 2.

In [ ]:
model.save_pretrained_gguf("operator_dryrun_gguf", tokenizer, quantization_method="q8_0")
!ls -la operator_dryrun_gguf

In [ ]:
import glob, shutil
candidates = [f for f in glob.glob("**/*.gguf", recursive=True) if "mmproj" not in f.lower()]
assert candidates, "No GGUF found -- check the export cell's output for errors"
gguf = max(candidates, key=os.path.getsize)  # the language model, not a vision projector
print("Using", gguf)
shutil.copy(gguf, "qwen3.5-0.8b-operator-q8_0.gguf")
if ON_KAGGLE:
    print("Download /kaggle/working/qwen3.5-0.8b-operator-q8_0.gguf from the Output tab")
else:
    from google.colab import files
    files.download("qwen3.5-0.8b-operator-q8_0.gguf")

## 8. On your machine

```powershell
# put the file in models/dryrun/, then:
llama-server -m models/dryrun/qwen3.5-0.8b-operator-q8_0.gguf --port 8080 --jinja `
    --chat-template-kwargs '{\"enable_thinking\": false}'

# held-out score (runs each answer in headless Blender):
python scripts/eval_operator_agent.py --base-url http://127.0.0.1:8080/v1

# inside Blender: install addon/blender_ai_dryrun.zip, open the N-panel "AI" tab, type a command
```